# LeetCode 1060 — Missing Element in Sorted Array bằng Python

Link đề: https://leetcode.com/problems/missing-element-in-sorted-array/ · Độ khó: **Medium** · Premium (cần tài khoản LeetCode Premium để xem đề) · Chủ đề: Array (mảng), Binary Search (tìm kiếm nhị phân)

> **Lưu ý:** đề bài của bài Premium này được viết theo hiểu biết và theo bản mô tả đã có; nên đối chiếu lại nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: cho mảng số nguyên `nums` **sắp xếp tăng dần**, các phần tử **khác nhau**, và số nguyên `k`. Trả về số **thứ `k` bị thiếu**, tính **bắt đầu từ số ngoài cùng bên trái của mảng** (tức là chỉ xét các số lớn hơn `nums[0]`). Ví dụ `nums = [4,7,9,10]`: các số bị thiếu là `5, 6, 8, 11, 12, ...`. Ý tưởng chính: số lượng số bị thiếu **trước** `nums[i]` là `nums[i] - nums[0] - i`, một hàm **không giảm** theo `i`, nên dùng **tìm kiếm nhị phân** trên `i`; câu hỏi nâng cao của đề yêu cầu `O(log n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và công thức "số lượng số bị thiếu trước `nums[i]`" là `missing(i) = nums[i] - nums[0] - i`.
2. Nắm các công cụ Python cần thiết: `set`, vòng lặp, `bisect_left` / `bisect_right` (kể cả tham số `key`), chia nguyên `//`.
3. Hiểu **tìm kiếm nhị phân tìm vị trí đầu tiên thỏa điều kiện đơn điệu** (`missing(i) >= k`), và cách suy ra đáp án từ vị trí tìm được.
4. Đi qua 4 cách: sinh từng số bị thiếu (brute force, dùng làm oracle), duyệt tuyến tính theo các khoảng trống, tìm kiếm nhị phân trên chỉ số (cách nộp chính) và tìm kiếm nhị phân trên giá trị đáp án.
5. Tránh các lỗi kinh điển: quên trường hợp đáp án nằm **sau** phần tử cuối, lệch một (`>` thay `>=`), quên trừ `nums[0]`, tính từ `1` thay vì từ `nums[0]`, quên trừ phần đã dùng khi duyệt khoảng trống, và vòng lặp `O(k)` quá chậm với `k = 10^8`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` được sắp xếp **tăng dần**, mọi phần tử **khác nhau**, và một số nguyên `k`. Trả về số bị thiếu thứ `k` tính **từ số ngoài cùng bên trái của mảng**.

**Ví dụ 1:** `nums = [4,7,9,10]`, `k = 1` -> `5`. Số bị thiếu đầu tiên là `5`.

**Ví dụ 2:** `nums = [4,7,9,10]`, `k = 3` -> `8`. Các số bị thiếu là `[5,6,8,...]` nên số thứ ba là `8`.

**Ví dụ 3:** `nums = [1,2,4]`, `k = 3` -> `6`. Các số bị thiếu là `[3,5,6,7,...]` nên số thứ ba là `6`.

**Ràng buộc:**

- `1 <= nums.length <= 5 * 10^4`
- `1 <= nums[i] <= 10^7`
- `nums` sắp xếp tăng dần, mọi phần tử khác nhau.
- `1 <= k <= 10^8`

**Câu hỏi nâng cao:** bạn có thể giải trong độ phức tạp thời gian logarit `O(log n)` không?

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
import random, time

examples = [
    ([4, 7, 9, 10], 1, 5),
    ([4, 7, 9, 10], 3, 8),
    ([1, 2, 4], 3, 6),
]
for nums, k, expected in examples:
    print(f"nums={nums} k={k} -> kỳ vọng {expected}")

nums=[4, 7, 9, 10] k=1 -> kỳ vọng 5
nums=[4, 7, 9, 10] k=3 -> kỳ vọng 8
nums=[1, 2, 4] k=3 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Các số bị thiếu và `missing(i)`

Với `nums = [4, 7, 9, 10]`, các số lớn hơn `nums[0] = 4` mà không có trong mảng là `5, 6, 8, 11, 12, ...`. Số lượng số bị thiếu **nhỏ hơn** `nums[i]` (và lớn hơn `nums[0]`) là

`missing(i) = nums[i] - nums[0] - i`:

Giải thích: các số nguyên nằm trong khoảng `(nums[0], nums[i]]` có đúng `nums[i] - nums[0]` số. Trong đó có `i` số thuộc mảng (`nums[1], ..., nums[i]`, vì mảng tăng chặt). Số còn lại là các số bị thiếu, và chúng đều nhỏ hơn `nums[i]`: `nums[i] - nums[0] - i`.

In [2]:
nums = [4, 7, 9, 10]
missing = [nums[i] - nums[0] - i for i in range(len(nums))]
print("missing(i) =", missing)
miss_list = [x for x in range(nums[0] + 1, nums[-1] + 1) if x not in set(nums)]
print("các số thiếu trong khoảng nums[0]..nums[-1]:", miss_list, "| số lượng:", len(miss_list), "= missing(n - 1) =", missing[-1])

missing(i) = [0, 2, 3, 3]
các số thiếu trong khoảng nums[0]..nums[-1]: [5, 6, 8] | số lượng: 3 = missing(n - 1) = 3


### 3.2. `missing(i)` không giảm

Từ `nums[i + 1] >= nums[i] + 1` ta có `missing(i + 1) = nums[i + 1] - nums[0] - i - 1 >= nums[i] - nums[0] - i = missing(i)`. Do đó `missing` là hàm **không giảm** theo `i`: có thể tìm kiếm nhị phân vị trí đầu tiên mà `missing(i) >= k`.

In [3]:
rng = random.Random(32)
for _ in range(1000):
    n = rng.randint(1, 10)
    nums = sorted(rng.sample(range(1, 40), n))
    m = [nums[i] - nums[0] - i for i in range(n)]
    assert m == sorted(m) and m[0] == 0
print("missing(i) luôn không giảm và missing(0) = 0 trên 1000 mảng ngẫu nhiên")

missing(i) luôn không giảm và missing(0) = 0 trên 1000 mảng ngẫu nhiên


### 3.3. `bisect` và tìm vị trí đầu tiên thỏa điều kiện

`bisect_left(a, x)` trả về chỉ số đầu tiên có `a[i] >= x` trong mảng **không giảm**. Từ Python 3.10 có thêm tham số `key` cho phép tìm kiếm theo một hàm của phần tử. Với `missing(i)` ta có thể viết `bisect_left(range(n), k, key=lambda i: nums[i] - nums[0] - i)`.

In [4]:
a = [0, 2, 3, 3]
print(bisect_left(a, 1), bisect_left(a, 3), bisect_right(a, 3), bisect_left(a, 4))
nums = [4, 7, 9, 10]
print("vị trí đầu tiên có missing >= 3:", bisect_left(range(len(nums)), 3, key=lambda i: nums[i] - nums[0] - i))

1 2 4 4
vị trí đầu tiên có missing >= 3: 2


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `nums = [4, 7, 9, 10]`, `k = 3`.

- `missing = [0, 2, 3, 3]`.
- Vị trí đầu tiên có `missing >= 3` là `i = 2` (`nums[2] = 9`, `missing = 3`).
- Vậy số bị thiếu thứ `3` nằm **ngay trước** `nums[2] = 9` và **sau** `nums[1] = 7`: nó cách `nums[1]` đúng `k - missing(1) = 3 - 2 = 1` bước ⇒ `7 + 1 = 8` ✓.

Ví dụ 3: `nums = [1, 2, 4]`, `k = 3`.

- `missing = [0, 0, 1]`, không có `i` nào có `missing >= 3`: đáp án nằm **sau** phần tử cuối `4`. Đã có `missing(2) = 1` số bị thiếu trước `4`; còn cần `3 - 1 = 2` số nữa sau `4`: `4 + 2 = 6` ✓.

Công thức chung: nếu `i` là vị trí đầu tiên có `missing(i) >= k` thì đáp án là `nums[i - 1] + (k - missing(i - 1))`; nếu không có `i` như vậy thì đáp án là `nums[-1] + (k - missing(n - 1))`.

## 5. Cách 1 — Brute Force: sinh từng số bị thiếu

**Ý tưởng.** Bắt đầu từ `x = nums[0]`; tăng `x` lên từng đơn vị, mỗi lần `x` không thuộc `nums` thì đếm một số bị thiếu; khi đếm đủ `k` thì trả về `x`. Đúng nguyên văn đề nên là **oracle độc lập**.

In [5]:
def missing_brute(nums: List[int], k: int) -> int:
    present = set(nums)
    x = nums[0]
    count = 0
    while True:
        x += 1
        if x not in present:
            count += 1
            if count == k:
                return x

for nums, k, expected in examples:
    got = missing_brute(nums, k)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 5 OK
brute = 8 OK
brute = 6 OK


**Độ phức tạp và nút thắt.** Thời gian `O(k + n)` (số lần lặp cỡ `k + n`): với `k = 10^8` quá chậm. Đo thử và ngoại suy:

In [6]:
nums = list(range(1, 50001, 2))                          # 25000 phần tử
for k in (10 ** 5, 10 ** 6, 3 * 10 ** 6):
    t0 = time.perf_counter()
    missing_brute(nums, k)
    dt = time.perf_counter() - t0
    print(f"k={k:8}: {dt * 1000:8.1f} ms -> ngoại suy k=10^8: ~{dt * 1e8 / k:6.1f} s")

k=  100000:     15.3 ms -> ngoại suy k=10^8: ~  15.3 s
k= 1000000:     93.5 ms -> ngoại suy k=10^8: ~   9.4 s


k= 3000000:    185.3 ms -> ngoại suy k=10^8: ~   6.2 s


Nút thắt: ta đi qua **từng số** trong khi chỉ cần biết số bị thiếu thứ `k` nằm trong **khoảng trống nào** giữa hai phần tử liên tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Khoảng trống.** Giữa `nums[i - 1]` và `nums[i]` có đúng `gap_i = nums[i] - nums[i - 1] - 1` số bị thiếu. Duyệt các khoảng trống, trừ dần `k`: khi `k <= gap_i` thì đáp án là `nums[i - 1] + k`. Nếu duyệt hết mà `k` vẫn còn thì đáp án là `nums[-1] + k`. Thời gian `O(n)`.

**Nhị phân.** `missing(i)` (số lượng số bị thiếu trước `nums[i]`) là hàm không giảm, nên ta tìm `i` nhỏ nhất có `missing(i) >= k` bằng tìm kiếm nhị phân `O(log n)`; đáp án nằm ngay trước `nums[i]`, sau `nums[i - 1]`, tại `nums[i - 1] + (k - missing(i - 1))`. Có thể rút gọn: vì `missing(i - 1) = nums[i - 1] - nums[0] - (i - 1)` nên `đáp án = nums[0] + k + i - 1` (và `nums[0] + k + n - 1` nếu không có `i`).

| Cách | Ý tưởng |
|---|---|
| 2 | Duyệt tuyến tính các khoảng trống, trừ dần `k` |
| 3 | Nhị phân trên chỉ số `i` với `missing(i) >= k` (nộp) |
| 4 | Nhị phân trên **giá trị** `x`: đếm số bị thiếu `<= x` bằng `bisect_right` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt tuyến tính các khoảng trống

**Giải thích từng đoạn.**

- Với mỗi cặp liên tiếp, `gap = nums[i] - nums[i - 1] - 1` là số số bị thiếu giữa chúng.
- Nếu `k <= gap`: số thứ `k` nằm trong khoảng trống này: `nums[i - 1] + k`.
- Ngược lại `k -= gap` (đã "dùng hết" khoảng trống này) và đi tiếp.
- Duyệt hết: đáp án sau phần tử cuối: `nums[-1] + k`.

In [7]:
def missing_gaps(nums: List[int], k: int) -> int:
    for i in range(1, len(nums)):
        gap = nums[i] - nums[i - 1] - 1
        if k <= gap:
            return nums[i - 1] + k
        k -= gap
    return nums[-1] + k

for nums, k, expected in examples:
    got = missing_gaps(nums, k)
    print(f"khoảng trống = {got}", "OK" if got == expected else "SAI")
    assert got == expected

khoảng trống = 5 OK
khoảng trống = 8 OK
khoảng trống = 6 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        for i in range(1, len(nums)):
            gap = nums[i] - nums[i - 1] - 1            # số bị thiếu giữa nums[i - 1] và nums[i]
            if k <= gap:
                return nums[i - 1] + k                 # số thứ k nằm trong khoảng trống này
            k -= gap                                   # dùng hết khoảng trống này
        return nums[-1] + k                            # nằm sau phần tử cuối

sol_gaps = lambda nums, k: Solution().missingElement(nums, k)
for nums, k, expected in examples:
    assert sol_gaps(nums, k) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(1)`.

---

### 7.2. Cách 3 — Nhị phân trên chỉ số (cách nộp chính)

**Giải thích từng đoạn.**

- `missing(i) = nums[i] - nums[0] - i`.
- Nếu `k > missing(n - 1)` thì đáp án sau phần tử cuối: `nums[-1] + (k - missing(n - 1))`.
- Ngược lại tìm `i` nhỏ nhất trong `[0, n - 1]` có `missing(i) >= k`: `lo, hi = 0, n - 1`; `mid`; nếu `missing(mid) >= k` thì `hi = mid` ngược lại `lo = mid + 1`.
- Khi `lo == hi` ta có `i = lo >= 1` (vì `missing(0) = 0 < k`). Đáp án `nums[i - 1] + (k - missing(i - 1))`.

In [9]:
def missing_binary(nums: List[int], k: int) -> int:
    n = len(nums)
    missing = lambda i: nums[i] - nums[0] - i
    if k > missing(n - 1):
        return nums[-1] + k - missing(n - 1)
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if missing(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return nums[lo - 1] + k - missing(lo - 1)

for nums, k, expected in examples:
    got = missing_binary(nums, k)
    print(f"nhị phân chỉ số = {got}", "OK" if got == expected else "SAI")
    assert got == expected

nhị phân chỉ số = 5 OK
nhị phân chỉ số = 8 OK
nhị phân chỉ số = 6 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        n = len(nums)

        def missing(i: int) -> int:                    # số lượng số bị thiếu trước nums[i]
            return nums[i] - nums[0] - i

        if k > missing(n - 1):                         # đáp án nằm sau phần tử cuối
            return nums[-1] + k - missing(n - 1)
        lo, hi = 0, n - 1                              # tìm i nhỏ nhất có missing(i) >= k
        while lo < hi:
            mid = (lo + hi) // 2
            if missing(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return nums[lo - 1] + k - missing(lo - 1)      # nằm giữa nums[lo - 1] và nums[lo]

sol_binary = lambda nums, k: Solution().missingElement(nums, k)
for nums, k, expected in examples:
    assert sol_binary(nums, k) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(log n)`, bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Nhị phân trên giá trị đáp án

**Ý tưởng khác.** Với mỗi giá trị `x >= nums[0]`, số lượng số bị thiếu trong `(nums[0], x]` bằng `(x - nums[0] + 1) - (số phần tử của nums <= x)`: tổng số nguyên trong `[nums[0], x]` trừ số phần tử của mảng trong đoạn đó. Hàm `f(x)` này **không giảm** theo `x`. Số bị thiếu thứ `k` chính là `x` **nhỏ nhất** có `f(x) >= k`. Cận trên: `nums[0] + k + n` (luôn đủ). Dùng `bisect_right(nums, x)` đếm phần tử `<= x`: mỗi bước `O(log n)`, tổng `O(log(k + n) * log n)`.

In [11]:
def missing_value_search(nums: List[int], k: int) -> int:
    def f(x: int) -> int:                              # số số bị thiếu trong [nums[0], x]
        return (x - nums[0] + 1) - bisect_right(nums, x)

    lo, hi = nums[0], nums[0] + k + len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, k, expected in examples:
    got = missing_value_search(nums, k)
    print(f"nhị phân giá trị = {got}", "OK" if got == expected else "SAI")
    assert got == expected

nhị phân giá trị = 5 OK
nhị phân giá trị = 8 OK
nhị phân giá trị = 6 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        from bisect import bisect_right

        def count_missing(x: int) -> int:              # số số bị thiếu trong [nums[0], x]
            return (x - nums[0] + 1) - bisect_right(nums, x)

        lo, hi = nums[0], nums[0] + k + len(nums)      # đáp án luôn nằm trong [nums[0], nums[0] + k + n]
        while lo < hi:
            mid = (lo + hi) // 2
            if count_missing(mid) >= k:                # x nhỏ nhất có ít nhất k số bị thiếu
                hi = mid
            else:
                lo = mid + 1
        return lo

sol_value = lambda nums, k: Solution().missingElement(nums, k)
for nums, k, expected in examples:
    assert sol_value(nums, k) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(log(k + n) * log n)`, bộ nhớ `O(1)`.

## 8. TRACE từng bước

### 8.1. Nhị phân trên chỉ số cho `nums = [4, 7, 9, 10]`, `k = 3`

In `lo`, `hi`, `mid`, `missing(mid)` ở mỗi bước.

In [13]:
def trace_binary(nums, k):
    n = len(nums)
    missing = lambda i: nums[i] - nums[0] - i
    print("nums =", nums, "| missing =", [missing(i) for i in range(n)], "| k =", k)
    if k > missing(n - 1):
        ans = nums[-1] + k - missing(n - 1)
        print(f"k > missing(n - 1) = {missing(n - 1)}: đáp án sau phần tử cuối = {nums[-1]} + ({k} - {missing(n - 1)}) = {ans}")
        return ans
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        print(f"  lo={lo} hi={hi} mid={mid} nums[mid]={nums[mid]} missing(mid)={missing(mid)}", "-> hi = mid" if missing(mid) >= k else "-> lo = mid + 1")
        if missing(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    ans = nums[lo - 1] + k - missing(lo - 1)
    print(f"i = {lo}: đáp án = nums[{lo - 1}] + (k - missing({lo - 1})) = {nums[lo - 1]} + ({k} - {missing(lo - 1)}) = {ans}")
    return ans

assert trace_binary([4, 7, 9, 10], 3) == 8
print()
assert trace_binary([1, 2, 4], 3) == 6

nums = [4, 7, 9, 10] | missing = [0, 2, 3, 3] | k = 3
  lo=0 hi=3 mid=1 nums[mid]=7 missing(mid)=2 -> lo = mid + 1
  lo=2 hi=3 mid=2 nums[mid]=9 missing(mid)=3 -> hi = mid
i = 2: đáp án = nums[1] + (k - missing(1)) = 7 + (3 - 2) = 8

nums = [1, 2, 4] | missing = [0, 0, 1] | k = 3
k > missing(n - 1) = 1: đáp án sau phần tử cuối = 4 + (3 - 1) = 6


### 8.2. Duyệt khoảng trống cho `nums = [4, 7, 9, 10]`, `k = 4`

In từng khoảng trống và giá trị `k` còn lại.

In [14]:
nums, k = [4, 7, 9, 10], 4
for i in range(1, len(nums)):
    gap = nums[i] - nums[i - 1] - 1
    print(f"khoảng ({nums[i - 1]}, {nums[i]}): {gap} số thiếu, k còn {k}", end=" ")
    if k <= gap:
        print(f"-> trả về {nums[i - 1]} + {k} = {nums[i - 1] + k}")
        break
    k -= gap
    print(f"-> trừ, k = {k}")
else:
    print(f"hết mảng: đáp án = nums[-1] + k = {nums[-1]} + {k} = {nums[-1] + k}")
assert missing_gaps([4, 7, 9, 10], 4) == 11 == missing_brute([4, 7, 9, 10], 4)

khoảng (4, 7): 2 số thiếu, k còn 4 -> trừ, k = 2
khoảng (7, 9): 1 số thiếu, k còn 2 -> trừ, k = 1
khoảng (9, 10): 0 số thiếu, k còn 1 -> trừ, k = 1
hết mảng: đáp án = nums[-1] + k = 10 + 1 = 11


### 8.3. Nhị phân trên giá trị cho `nums = [4, 7, 9, 10]`, `k = 3`

In đoạn `[lo, hi]` và `f(mid)`.

In [15]:
nums, k = [4, 7, 9, 10], 3
f = lambda x: (x - nums[0] + 1) - bisect_right(nums, x)
lo, hi = nums[0], nums[0] + k + len(nums)
print("f(x) cho x = 4..12:", {x: f(x) for x in range(4, 13)})
while lo < hi:
    mid = (lo + hi) // 2
    print(f"lo={lo} hi={hi} mid={mid} f(mid)={f(mid)}", "-> hi = mid" if f(mid) >= k else "-> lo = mid + 1")
    if f(mid) >= k:
        hi = mid
    else:
        lo = mid + 1
print("đáp án =", lo)
assert lo == 8

f(x) cho x = 4..12: {4: 0, 5: 1, 6: 2, 7: 2, 8: 3, 9: 3, 10: 3, 11: 4, 12: 5}
lo=4 hi=11 mid=7 f(mid)=2 -> lo = mid + 1
lo=8 hi=11 mid=9 f(mid)=3 -> hi = mid
lo=8 hi=9 mid=8 f(mid)=3 -> hi = mid
đáp án = 8


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (nhị phân trên chỉ số): `O(log n)` thời gian, đúng yêu cầu câu hỏi nâng cao.

```python
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        n = len(nums)

        def missing(i: int) -> int:                    # số lượng số bị thiếu trước nums[i]
            return nums[i] - nums[0] - i

        if k > missing(n - 1):                         # đáp án nằm sau phần tử cuối
            return nums[-1] + k - missing(n - 1)
        lo, hi = 0, n - 1                              # tìm i nhỏ nhất có missing(i) >= k
        while lo < hi:
            mid = (lo + hi) // 2
            if missing(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return nums[lo - 1] + k - missing(lo - 1)      # nằm giữa nums[lo - 1] và nums[lo]
```

In [16]:
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        n = len(nums)

        def missing(i: int) -> int:                    # số lượng số bị thiếu trước nums[i]
            return nums[i] - nums[0] - i

        if k > missing(n - 1):                         # đáp án nằm sau phần tử cuối
            return nums[-1] + k - missing(n - 1)
        lo, hi = 0, n - 1                              # tìm i nhỏ nhất có missing(i) >= k
        while lo < hi:
            mid = (lo + hi) // 2
            if missing(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return nums[lo - 1] + k - missing(lo - 1)      # nằm giữa nums[lo - 1] và nums[lo]

sol = lambda nums, k: Solution().missingElement(nums, k)
for nums, k, expected in examples:
    got = sol(nums, k)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

5 OK
8 OK
6 OK


## 10. Vì sao đúng

**Công thức `missing(i)`.** Các số nguyên trong `(nums[0], nums[i]]` là `nums[i] - nums[0]` số; trong đó có đúng `i` số thuộc mảng (`nums[1], ..., nums[i]`, nhờ các phần tử khác nhau và sắp tăng). Vậy số số **bị thiếu** trong khoảng này (đều nhỏ hơn `nums[i]`) là `nums[i] - nums[0] - i = missing(i)`.

**Đơn điệu.** `missing(i + 1) - missing(i) = nums[i + 1] - nums[i] - 1 >= 0` (vì `nums[i + 1] > nums[i]`). Cũng có `missing(0) = 0`.

**Cách 3.** Gọi `i` là chỉ số nhỏ nhất có `missing(i) >= k`.

- Nếu `k > missing(n - 1)` thì mọi số bị thiếu trong `(nums[0], nums[-1])` là `missing(n - 1) < k` số; số thứ `k` nằm sau `nums[-1]`, mọi số lớn hơn `nums[-1]` đều bị thiếu, nên đáp án là `nums[-1] + (k - missing(n - 1))`.
- Ngược lại `i` tồn tại và `i >= 1` (vì `missing(0) = 0 < k`). Có `missing(i - 1) < k <= missing(i)` số bị thiếu nhỏ hơn `nums[i - 1]`, nên số thứ `k` lớn hơn `nums[i - 1]`; và có `missing(i) >= k` số bị thiếu nhỏ hơn `nums[i]` nên số thứ `k` nhỏ hơn `nums[i]`. Giữa `nums[i - 1]` và `nums[i]` mọi số đều bị thiếu, nên số thứ `k` chính là số thứ `k - missing(i - 1)` sau `nums[i - 1]`, tức `nums[i - 1] + k - missing(i - 1)`.

**Cách 2** là cùng ý tưởng nhưng duyệt từng khoảng trống: `k -= gap` mô phỏng việc tính `k - missing(i - 1)`.

**Cách 4.** `f(x) = (x - nums[0] + 1) - #{a in nums : a <= x}` là số số nguyên trong `[nums[0], x]` không thuộc mảng, nên `f(nums[0]) = 0`, `f` không giảm và tăng đúng 1 tại mỗi `x` bị thiếu. Số bị thiếu thứ `k` là `x` nhỏ nhất có `f(x) = k`, cũng là nhỏ nhất có `f(x) >= k`. Cận trên `nums[0] + k + n` thỏa `f >= k` vì có tối đa `n` phần tử trong mảng.

**Kiểm chứng bằng máy.** Oracle (Cách 1) sinh từng số bị thiếu. Ta đối chiếu mọi cách với oracle trên **mọi** mảng con của `{1..9}` (khác rỗng) với `k` từ `1` đến `12`, cộng nhiều test ngẫu nhiên có seed.

In [17]:
from itertools import combinations
impls = {"khoảng trống": missing_gaps, "nhị phân chỉ số": missing_binary, "nhị phân giá trị": missing_value_search,
         "nộp (Cách 3)": sol, "Cách 2 (nộp)": sol_gaps, "Cách 4 (nộp)": sol_value}
total = 0
for size in range(1, 10):
    for comb in combinations(range(1, 10), size):
        nums = list(comb)
        for k in range(1, 13):
            ref = missing_brute(nums, k)
            for name, fn in impls.items():
                assert fn(nums, k) == ref, (name, nums, k, fn(nums, k), ref)
            total += 1
print("đã đối chiếu", total, "cặp (nums, k): mọi cách khớp oracle")

đã đối chiếu 6132 cặp (nums, k): mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Sinh từng số bị thiếu | `O(k + n)` | `O(n)` |
| 2. Duyệt khoảng trống | `O(n)` | `O(1)` |
| 3. Nhị phân chỉ số | `O(log n)` | `O(1)` |
| 4. Nhị phân giá trị | `O(log(k + n) * log n)` | `O(1)` |

Đo ở cỡ tối đa (`n = 5 * 10^4`, giá trị tới `10^7`, `k` tới `10^8`) với các dạng: ngẫu nhiên thưa, mảng liên tiếp, một phần tử, đáp án nằm sau phần tử cuối:

In [18]:
rng = random.Random(11)
cases = {
    "ngẫu nhiên thưa": (sorted(rng.sample(range(1, 10 ** 7 + 1), 5 * 10 ** 4)), 10 ** 6),
    "liên tiếp 1..50000, k = 10^8": (list(range(1, 50001)), 10 ** 8),
    "một phần tử 10^7, k = 10^8": ([10 ** 7], 10 ** 8),
    "hai phần tử 1 và 10^7, k nằm trong khoảng trống": ([1, 10 ** 7], 9999999 - 1),
    "ngẫu nhiên, k = 1": (sorted(rng.sample(range(1, 10 ** 7 + 1), 5 * 10 ** 4)), 1),
    "ngẫu nhiên, k nhỏ vừa": (sorted(rng.sample(range(1, 10 ** 7 + 1), 5 * 10 ** 4)), 9 * 10 ** 6),
}
for label, (nums, k) in cases.items():
    ref = sol(nums, k)
    row = [f"{label:48}"]
    for name, fn in (("khoảng trống", sol_gaps), ("nhị phân chỉ số", sol), ("nhị phân giá trị", sol_value)):
        t0 = time.perf_counter()
        got = fn(nums, k)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.3f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên thưa                                  | khoảng trống   0.006 ms | nhị phân chỉ số   0.005 ms | nhị phân giá trị   0.005 ms | đáp án 1004993
liên tiếp 1..50000, k = 10^8                     | khoảng trống   0.002 ms | nhị phân chỉ số   0.001 ms | nhị phân giá trị   0.001 ms | đáp án 100050000
một phần tử 10^7, k = 10^8                       | khoảng trống   0.002 ms | nhị phân chỉ số   0.001 ms | nhị phân giá trị   0.001 ms | đáp án 110000000
hai phần tử 1 và 10^7, k nằm trong khoảng trống  | khoảng trống   0.001 ms | nhị phân chỉ số   0.001 ms | nhị phân giá trị   0.001 ms | đáp án 9999999
ngẫu nhiên, k = 1                                | khoảng trống   0.004 ms | nhị phân chỉ số   0.004 ms | nhị phân giá trị   0.004 ms | đáp án 27
ngẫu nhiên, k nhỏ vừa                            | khoảng trống   0.005 ms | nhị phân chỉ số   0.005 ms | nhị phân giá trị   0.004 ms | đáp án 9045609


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), test ngẫu nhiên có seed (đúng ràng buộc) và các tính chất.

In [19]:
def check_all(nums, k, expected=None, brute=True):
    results = {name: fn(nums, k) for name, fn in impls.items()}
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (name, got, first, nums[:10], k)
    if brute and k + len(nums) <= 3 * 10 ** 5:
        assert missing_brute(nums, k) == first, (nums[:10], k)
    if expected is not None:
        assert first == expected, (nums[:10], k, first, expected)
    return first

# 12.1 ví dụ đề
for nums, k, ex in examples:
    check_all(nums, k, ex)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], 1, 2),                                   # một phần tử: số thiếu đầu tiên là nums[0] + 1
    ([1], 5, 6),                                   # một phần tử: nums[0] + k
    ([5], 1, 6),                                   # không bắt đầu từ 1: chỉ đếm các số lớn hơn nums[0]
    ([1, 2, 3], 1, 4),                             # liên tiếp: số thiếu đầu tiên sau phần tử cuối
    ([1, 3], 1, 2),                                # khoảng trống một số
    ([1, 3], 2, 4),                                # số thứ hai nằm sau phần tử cuối
    ([1, 3], 3, 5),
    ([4, 7, 9, 10], 4, 11),                        # số thiếu: 5, 6, 8, 11
    ([4, 7, 9, 10], 5, 12),
    ([1, 2, 4], 1, 3),
    ([1, 2, 4], 2, 5),
    ([2, 5, 6, 9], 4, 8),                          # tính tay ở bài tập 2: thiếu 3, 4, 7, 8
    ([10 ** 7], 10 ** 8, 10 ** 7 + 10 ** 8),       # giá trị và k lớn nhất
    (list(range(1, 50001)), 10 ** 8, 50000 + 10 ** 8),   # mảng dài liên tiếp, k lớn nhất
    ([1, 10 ** 7], 1, 2),                          # khoảng trống rất lớn: số đầu tiên
    ([1, 10 ** 7], 9999998, 9999999),              # số cuối cùng trong khoảng trống
    ([1, 10 ** 7], 9999999, 10 ** 7 + 1),          # ngay sau phần tử cuối
    ([3, 4, 5, 8], 2, 7),                          # thiếu 6, 7: số thứ hai là 7
]
for nums, k, ex in edge:
    check_all(nums, k, ex)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 18 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_no_tail(nums, k):                        # quên trường hợp đáp án nằm sau phần tử cuối
    n = len(nums)
    missing = lambda i: nums[i] - nums[0] - i
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if missing(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return nums[lo - 1] + k - missing(lo - 1) if lo >= 1 else nums[0] + k

def wrong_strict(nums, k):                         # lệch một: dùng missing(mid) > k thay vì >=
    n = len(nums)
    missing = lambda i: nums[i] - nums[0] - i
    if k > missing(n - 1):
        return nums[-1] + k - missing(n - 1)
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if missing(mid) > k:
            hi = mid
        else:
            lo = mid + 1
    return nums[lo - 1] + k - missing(lo - 1) if lo >= 1 else nums[0] + k

def wrong_no_first(nums, k):                       # quên trừ nums[0]: coi mảng bắt đầu từ 0
    n = len(nums)
    missing = lambda i: nums[i] - i
    if k > missing(n - 1):
        return nums[-1] + k - missing(n - 1)
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if missing(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return nums[lo - 1] + k - missing(lo - 1) if lo >= 1 else nums[0] + k

def wrong_from_one(nums, k):                       # tính từ số nguyên dương 1 thay vì từ nums[0] (bài khác: Kth Missing Positive)
    present = set(nums)
    x = 0
    while k:
        x += 1
        if x not in present:
            k -= 1
    return x

def wrong_no_subtract(nums, k):                    # duyệt khoảng trống nhưng quên trừ khoảng đã dùng
    for i in range(1, len(nums)):
        gap = nums[i] - nums[i - 1] - 1
        if k <= gap:
            return nums[i - 1] + k
    return nums[-1] + k

def find_counter(wrong, tries=30000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 5)
        nums = sorted(rng.sample(range(1, 12), n))
        k = rng.randint(1, 8)
        try:
            bad = wrong(nums, k) != missing_brute(nums, k)
        except Exception:
            bad = True
        if bad:
            return nums, k
    return None

counters = {}
for name, fn in (("quên trường hợp đáp án sau phần tử cuối", wrong_no_tail), ("dùng > thay vì >= (lệch một)", wrong_strict),
                 ("quên trừ nums[0]", wrong_no_first), ("tính từ số 1 thay vì từ nums[0]", wrong_from_one),
                 ("quên trừ khoảng trống đã dùng", wrong_no_subtract)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    nums, k = case
    try:
        shown = fn(nums, k)
    except Exception as e:
        shown = type(e).__name__
    print(f"{name:42} nums={nums} k={k}: sai -> {shown}, đúng -> {missing_brute(nums, k)}")
    check_all(nums, k)
print("12.3 phản ví dụ: OK")

quên trường hợp đáp án sau phần tử cuối    nums=[1, 2, 6, 9, 10] k=7: sai -> 11, đúng -> 12
dùng > thay vì >= (lệch một)               nums=[1, 3, 7, 8] k=1: sai -> 3, đúng -> 2
quên trừ nums[0]                           nums=[3, 5, 6, 10] k=3: sai -> 6, đúng -> 8
tính từ số 1 thay vì từ nums[0]            nums=[3, 5, 6, 10] k=3: sai -> 4, đúng -> 8
quên trừ khoảng trống đã dùng              nums=[3, 5, 6, 10] k=3: sai -> 9, đúng -> 8
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: nhiều dải giá trị và k
rng = random.Random(4060)
for _ in range(3000):
    hi = rng.choice([10, 30, 100, 10 ** 4])
    n = rng.randint(1, min(12, hi))
    nums = sorted(rng.sample(range(1, hi + 1), n))
    check_all(nums, rng.randint(1, rng.choice([3, 20, 200])))
print("12.4 ngẫu nhiên nhỏ: OK")

# 12.5 ngẫu nhiên vừa (so oracle) và lớn (so các cách nhanh với nhau)
for _ in range(40):
    n = rng.randint(100, 2000)
    nums = sorted(rng.sample(range(1, 20000), n))
    check_all(nums, rng.randint(1, 10 ** 5))
for _ in range(40):
    n = rng.randint(1000, 5 * 10 ** 4)
    nums = sorted(rng.sample(range(1, 10 ** 7 + 1), n))
    check_all(nums, rng.randint(1, 10 ** 8), brute=False)
print("12.5 ngẫu nhiên vừa và lớn: OK")

# 12.6 tính chất: kết quả tăng chặt theo k, không thuộc nums, và đếm số thiếu đúng là k
for _ in range(300):
    n = rng.randint(1, 12)
    nums = sorted(rng.sample(range(1, 60), n))
    S = set(nums)
    prev = nums[0]
    for k in range(1, 40):
        ans = sol(nums, k)
        assert ans > prev and ans not in S                                        # tăng chặt và đúng là số thiếu
        assert sum(1 for x in range(nums[0] + 1, ans + 1) if x not in S) == k     # có đúng k số thiếu trong (nums[0], ans]
        prev = ans
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: OK


12.5 ngẫu nhiên vừa và lớn: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp đáp án nằm sau phần tử cuối

Nếu `k > missing(n - 1)` thì không có `i` nào thỏa `missing(i) >= k`; nhị phân khi đó trả về `n - 1` cho kết quả sai.

In [22]:
nums, k = counters["quên trường hợp đáp án sau phần tử cuối"]
print("nums =", nums, "k =", k, "| sai ->", wrong_no_tail(nums, k), "| đúng ->", sol(nums, k))
assert wrong_no_tail(nums, k) != sol(nums, k)
print("ví dụ 3 của đề (cần nhánh này):", wrong_no_tail([1, 2, 4], 3), "(đúng là 6)")

nums = [1, 2, 6, 9, 10] k = 7 | sai -> 11 | đúng -> 12
ví dụ 3 của đề (cần nhánh này): 5 (đúng là 6)


Vì sao sai: khi `k > missing(n - 1)` đáp án là `nums[-1] + (k - missing(n - 1))`. **Đúng:** kiểm tra riêng trường hợp này trước khi nhị phân.

### Lỗi 2 — Lệch một: dùng `>` thay `>=`

In [23]:
nums, k = counters["dùng > thay vì >= (lệch một)"]
print("nums =", nums, "k =", k, "| sai ->", wrong_strict(nums, k), "| đúng ->", sol(nums, k))
assert wrong_strict(nums, k) != sol(nums, k)

nums = [1, 3, 7, 8] k = 1 | sai -> 3 | đúng -> 2


Vì sao sai: khi `missing(i) == k` thì số thiếu thứ `k` đã nằm **trước** `nums[i]`, nên `i` hợp lệ. **Đúng:** điều kiện `missing(mid) >= k` và `hi = mid`.

### Lỗi 3 — Quên trừ `nums[0]`

In [24]:
nums, k = counters["quên trừ nums[0]"]
print("nums =", nums, "k =", k, "| sai ->", wrong_no_first(nums, k), "| đúng ->", sol(nums, k))
assert wrong_no_first(nums, k) != sol(nums, k)

nums = [3, 5, 6, 10] k = 3 | sai -> 6 | đúng -> 8


Vì sao sai: `nums[i] - i` chỉ đúng khi `nums[0] = 0`; ở đây phải trừ cả `nums[0]` vì ta chỉ đếm số bị thiếu **sau** `nums[0]`. **Đúng:** `missing(i) = nums[i] - nums[0] - i`.

### Lỗi 4 — Tính từ số `1` thay vì từ `nums[0]`

Đề này đếm bắt đầu từ số **ngoài cùng bên trái của mảng**; bài Kth Missing Positive Number (1539) mới đếm từ `1`.

In [25]:
nums, k = counters["tính từ số 1 thay vì từ nums[0]"]
print("nums =", nums, "k =", k, "| sai ->", wrong_from_one(nums, k), "| đúng ->", sol(nums, k))
assert wrong_from_one(nums, k) != sol(nums, k)
print("ví dụ 1 của đề:", wrong_from_one([4, 7, 9, 10], 1), "(đúng là 5)")

nums = [3, 5, 6, 10] k = 3 | sai -> 4 | đúng -> 8
ví dụ 1 của đề: 1 (đúng là 5)


Vì sao sai: các số nhỏ hơn `nums[0]` không được tính. **Đúng:** bắt đầu đếm từ `nums[0] + 1`.

### Lỗi 5 — Duyệt khoảng trống nhưng quên trừ phần đã dùng

In [26]:
nums, k = counters["quên trừ khoảng trống đã dùng"]
print("nums =", nums, "k =", k, "| sai ->", wrong_no_subtract(nums, k), "| đúng ->", sol(nums, k))
assert wrong_no_subtract(nums, k) != sol(nums, k)

nums = [3, 5, 6, 10] k = 3 | sai -> 9 | đúng -> 8


Vì sao sai: sau khi bỏ qua một khoảng trống đã dùng hết `gap` số, số cần tìm còn `k - gap` số nữa. **Đúng:** `k -= gap` sau mỗi khoảng trống.

### Lỗi 6 — Vòng lặp `O(k)` với `k` tới `10^8`

Cách sinh từng số (Cách 1) quá chậm, như đã đo ở mục 5. **Đúng:** nhảy theo khoảng trống hoặc nhị phân.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "tìm phần tử thứ `k` bị thiếu / thứ `k` không thuộc tập" trên mảng **đã sắp xếp** ⇒ tìm một đại lượng **đơn điệu** theo chỉ số (`missing(i) = nums[i] - nums[0] - i`) rồi **nhị phân**.
- Tìm vị trí **đầu tiên** thỏa điều kiện đơn điệu: `lo, hi = 0, n - 1`; `if cond(mid): hi = mid else: lo = mid + 1`.
- Luôn tách riêng trường hợp đáp án **ngoài mảng** (sau phần tử cuối).
- Bài biến thể nổi tiếng: Kth Missing Positive Number (1539): `missing(i) = nums[i] - (i + 1)`.
- Nhị phân trên **giá trị** (đếm bằng `bisect`) cũng là một cách đơn giản và khó sai.
- Cùng họ: Kth Missing Positive Number (1539), Find Peak Element (162), Search in Rotated Sorted Array (33).
- Test nên có: một phần tử, mảng liên tiếp, đáp án sau phần tử cuối, đáp án ngay trước phần tử `i`, giá trị lớn nhất.

## 15. Biến thể

1. **Kth Missing Positive Number (1539):** tìm số nguyên dương thứ `k` bị thiếu (đếm từ `1`): bài tập 5.
2. **Nhiều truy vấn `k`:** dựng mảng `missing` một lần rồi trả lời mỗi truy vấn bằng `bisect_left` trong `O(log n)`.
3. **Mảng có thể có phần tử trùng:** khử trùng trước (hoặc dùng `set`).
4. Bài liên quan: 1539, 162.

Kiểm biến thể 2 (nhiều truy vấn) bằng oracle:

In [27]:
def answer_queries(nums: List[int], ks: List[int]) -> List[int]:
    n = len(nums)
    miss = [nums[i] - nums[0] - i for i in range(n)]               # không giảm
    res = []
    for k in ks:
        i = bisect_left(miss, k)                                   # i nhỏ nhất có miss[i] >= k (hoặc n nếu không có)
        res.append(nums[0] + k + i - 1)                            # công thức rút gọn
    return res

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 10)
    nums = sorted(rng.sample(range(1, 50), n))
    ks = [rng.randint(1, 60) for _ in range(10)]
    assert answer_queries(nums, ks) == [missing_brute(nums, k) for k in ks], (nums, ks)
print("biến thể nhiều truy vấn: nums[0] + k + i - 1 với i = bisect_left(missing, k) khớp oracle")

biến thể nhiều truy vấn: nums[0] + k + i - 1 với i = bisect_left(missing, k) khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh từng số thiếu | Đếm từng `x` | `O(k + n)` | `O(n)` | Oracle |
| 2. Duyệt khoảng trống | Trừ dần `k` | `O(n)` | `O(1)` | Đơn giản, đủ tốt khi `n` nhỏ |
| 3. Nhị phân chỉ số | `missing(i) >= k` | `O(log n)` | `O(1)` | **Cách nộp chính**, đạt câu hỏi nâng cao |
| 4. Nhị phân giá trị | Đếm số thiếu `<= x` bằng `bisect` | `O(log(k + n) log n)` | `O(1)` | Khó sai, dễ nhớ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
nums = [2, 5, 6, 9]
print([nums[i] - nums[0] - i for i in range(len(nums))])
print(Solution().missingElement(nums, 1), Solution().missingElement(nums, 3))
print(bisect_left([0, 2, 3, 3], 3), bisect_left([0, 2, 3, 3], 4))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # danh sách missing
my_guess_2 = None      # hai kết quả missingElement
my_guess_3 = None      # hai kết quả bisect_left

### Bài 2 — Tính tay (dễ-vừa)

`nums = [2, 5, 6, 9]`, `k = 4`. Liệt kê các số bị thiếu và ghi số thứ `4`.

In [29]:
# Bài 2: điền kết quả tính tay
hand_missing_list, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành nhị phân trên chỉ số.

In [30]:
def exercise_3(nums: List[int], k: int) -> int:
    n = len(nums)
    missing = lambda i: nums[i] - nums[0] - i
    # TODO 1: nếu k > missing(n - 1) thì đáp án = nums[-1] + k - missing(n - 1)
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # TODO 2: nếu missing(mid) >= k thì hi = mid, ngược lại lo = mid + 1
        pass
    # TODO 3: trả về nums[lo - 1] + k - missing(lo - 1)
    pass


# for nums, k, ex in examples:
#     assert exercise_3(nums, k) == ex

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(nums, k)` bằng **nhị phân trên giá trị** (dùng `bisect_right` để đếm).

In [31]:
def exercise_4(nums: List[int], k: int) -> int:
    # TODO
    pass


# for nums, k, ex in examples:
#     assert exercise_4(nums, k) == ex

### Bài 5 — Mini challenge: Kth Missing Positive Number (khó)

Cho mảng số nguyên dương `arr` sắp xếp tăng dần, các phần tử khác nhau, và `k`. Trả về số nguyên dương **thứ `k` bị thiếu** (đếm bắt đầu từ `1`, không phải từ `arr[0]`). Ví dụ:

```python
exercise_5([2, 3, 4, 7, 11], 5)  -> 9    # thiếu 1, 5, 6, 8, 9, 10, ...
exercise_5([1, 2, 3, 4], 2)      -> 6    # thiếu 5, 6, ...
exercise_5([5], 3)               -> 3    # thiếu 1, 2, 3, 4, ...
```
Gợi ý: số số thiếu trước `arr[i]` là `arr[i] - (i + 1)`; dùng nhị phân như Cách 3.

In [32]:
def exercise_5(arr: List[int], k: int) -> int:
    # TODO
    pass


def positive_missing_oracle(arr, k):
    present, x = set(arr), 0
    while k:
        x += 1
        if x not in present:
            k -= 1
    return x


# assert exercise_5([2, 3, 4, 7, 11], 5) == 9
# assert exercise_5([1, 2, 3, 4], 2) == 6
# assert exercise_5([5], 3) == 3

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `missing(i) = nums[i] - nums[0] - i`.
- Mức 2: `nums = [2, 5, 6, 9]`: `5 - 2 - 1`, `6 - 2 - 2`, `9 - 2 - 3`.
- Mức 3: `[0, 2, 2, 4]`; `3` và `7` (các số thiếu `3, 4, 7, 8, ...`: số thứ nhất là `3`, số thứ ba là `7`); `2` và `4` (`bisect_left([0,2,3,3], 3) = 2`, và `4` vượt quá mọi phần tử nên bằng `len = 4`).

**Bài 2**
- Mức 1: các số thiếu nằm giữa `2` và `5`, giữa `6` và `9`, rồi sau `9`.
- Mức 2: `3, 4` (giữa `2` và `5`), `7, 8` (giữa `6` và `9`), rồi `10, 11, ...`.
- Mức 3: danh sách `[3, 4, 7, 8, 10, ...]`, số thứ `4` là `8`.

**Bài 3**
- Mức 1: `if k > missing(n - 1): return nums[-1] + k - missing(n - 1)`.
- Mức 2: `if missing(mid) >= k: hi = mid` `else: lo = mid + 1`.
- Mức 3: `return nums[lo - 1] + k - missing(lo - 1)`.

**Bài 4**
- Mức 1: `count_missing(x) = (x - nums[0] + 1) - bisect_right(nums, x)`.
- Mức 2: `lo, hi = nums[0], nums[0] + k + len(nums)`.
- Mức 3: `x` nhỏ nhất có `count_missing(x) >= k`: `if count_missing(mid) >= k: hi = mid else: lo = mid + 1`; trả `lo`.

**Bài 5**
- Mức 1: số số thiếu trước `arr[i]` là `arr[i] - (i + 1)`.
- Mức 2: nếu `k > arr[-1] - len(arr)` thì đáp án `arr[-1] + k - (arr[-1] - len(arr))`; ngược lại nhị phân tìm `i` đầu tiên có `arr[i] - (i + 1) >= k`.
- Mức 3: đáp án `i + k` (với `i` là chỉ số đầu tiên đó; hoặc `len(arr) + k` nếu không có).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
nums = [2, 5, 6, 9]
r1 = [nums[i] - nums[0] - i for i in range(len(nums))]
r2 = (Solution().missingElement(nums, 1), Solution().missingElement(nums, 3))
r3 = (bisect_left([0, 2, 3, 3], 3), bisect_left([0, 2, 3, 3], 4))
print(r1, r2, r3)
assert r1 == [0, 2, 2, 4]
assert r2 == (3, 7)
assert r3 == (2, 4)

[0, 2, 2, 4] (3, 7) (2, 4)


### Đáp án Bài 2

Giữa `2` và `5`: `3, 4`; giữa `6` và `9`: `7, 8`; sau `9`: `10, 11, ...`. Danh sách `[3, 4, 7, 8, 10, 11, ...]`, số thứ `4` là `8`.

In [34]:
hand_missing_list_solution, hand_answer_solution = [3, 4, 7, 8, 10], 8
nums = [2, 5, 6, 9]
assert hand_missing_list_solution == [x for x in range(3, 11) if x not in set(nums)][:5]
assert hand_answer_solution == hand_missing_list_solution[3] == sol(nums, 4) == missing_brute(nums, 4)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(nums: List[int], k: int) -> int:
    n = len(nums)
    missing = lambda i: nums[i] - nums[0] - i
    if k > missing(n - 1):
        return nums[-1] + k - missing(n - 1)
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if missing(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return nums[lo - 1] + k - missing(lo - 1)

for nums, k, ex in examples:
    assert exercise_3_solution(nums, k) == ex
rng = random.Random(3)
for _ in range(500):
    nums = sorted(rng.sample(range(1, 40), rng.randint(1, 10)))
    k = rng.randint(1, 30)
    assert exercise_3_solution(nums, k) == missing_brute(nums, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(nums: List[int], k: int) -> int:
    count_missing = lambda x: (x - nums[0] + 1) - bisect_right(nums, x)
    lo, hi = nums[0], nums[0] + k + len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if count_missing(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, k, ex in examples:
    assert exercise_4_solution(nums, k) == ex
rng = random.Random(4)
for _ in range(500):
    nums = sorted(rng.sample(range(1, 40), rng.randint(1, 10)))
    k = rng.randint(1, 30)
    assert exercise_4_solution(nums, k) == missing_brute(nums, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(arr: List[int], k: int) -> int:
    n = len(arr)
    lo, hi = 0, n                                   # i nhỏ nhất có arr[i] - (i + 1) >= k; n nếu không có
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo + k

def positive_missing_oracle_solution(arr, k):
    present, x = set(arr), 0
    while k:
        x += 1
        if x not in present:
            k -= 1
    return x

assert exercise_5_solution([2, 3, 4, 7, 11], 5) == 9
assert exercise_5_solution([1, 2, 3, 4], 2) == 6
assert exercise_5_solution([5], 3) == 3
rng = random.Random(5)
for _ in range(1000):
    arr = sorted(rng.sample(range(1, 40), rng.randint(1, 10)))
    k = rng.randint(1, 40)
    assert exercise_5_solution(arr, k) == positive_missing_oracle_solution(arr, k), (arr, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (nhị phân vị trí đầu tiên có `missing(i) >= k`)

```python
class Solution:
    def missingElement(self, nums: List[int], k: int) -> int:
        n = len(nums)
        missing = lambda i: nums[i] - nums[0] - i     # số bị thiếu trước nums[i]
        if k > missing(n - 1):                        # đáp án sau phần tử cuối
            return nums[-1] + k - missing(n - 1)
        lo, hi = 0, n - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if missing(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return nums[lo - 1] + k - missing(lo - 1)     # hoặc nums[0] + k + lo - 1
```

Công thức nhớ nhanh: **`missing(i) = nums[i] - nums[0] - i`; `i` đầu tiên có `missing(i) >= k`; đáp án `nums[i - 1] + k - missing(i - 1)`; nếu không có `i` thì `nums[-1] + k - missing(n - 1)`**.

### Checklist trước khi submit

- [ ] `missing(i)` trừ cả `nums[0]` và `i` (không phải `nums[i] - i`)?
- [ ] Xử lý trường hợp `k > missing(n - 1)` (đáp án sau phần tử cuối)?
- [ ] Điều kiện nhị phân là `>=` và `hi = mid`?
- [ ] Đáp án dùng `nums[lo - 1]` (có `lo >= 1` vì `missing(0) = 0 < k`)?
- [ ] Không dùng vòng lặp `O(k)` với `k` tới `10^8`?
- [ ] Đã thử: một phần tử, mảng liên tiếp, đáp án ngay sau `nums[0]`, đáp án sau phần tử cuối, `k = 10^8`?